# 16 Resolution Time ML

This copy notebook focuses on **time-to-resolution inference**.

We do not train a main model for `resolution_bool` here. Whether a case closed is useful metadata, but the modeling question for this stage is the length of resolution time among events with an observed closure.

Important treatment:
- `resolution = NA` means open / unresolved / censored
- unresolved events are summarized as context
- unresolved events are excluded from the regression target because no final resolution time exists

Split rule:
- one classic stratified `75/15/10` train/validation/test split
- stratification uses resolution-time bands plus `mayoral_administration` when feasible
- model selection is a hyperparameter-grid sweep on train/validation; test is untouched final evaluation

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
from project_name.modeling_stage import (
    RESOLUTION_BIAS_DIAGNOSTICS_PATH,
    RESOLUTION_FEATURE_IMPORTANCE_PATH,
    RESOLUTION_LEAKAGE_AUDIT_PATH,
    RESOLUTION_PREDICTIONS_PATH,
    RESOLUTION_TIME_RESULTS_PATH,
    compact_modeling_summary,
    feature_catalog,
    load_modeling_frame,
    run_resolution_time_ml,
    summarize_feature_groups,
)

observed = load_modeling_frame(view_name="strict_main", include_geometry=False, observed_only=True)
closed = observed[observed["resolution"].notna()].copy()
open_or_censored = observed[observed["resolution"].isna()].copy()

print(f"observed flood events: {len(observed):,}")
print(f"closed with resolution time: {len(closed):,}")
print(f"open / unresolved / censored: {len(open_or_censored):,}")
display(summarize_feature_groups(observed))
display(feature_catalog(observed).head(40))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
observed["resolution_bool"].astype("boolean").value_counts(dropna=False).plot.bar(ax=axes[0], color=["#2563EB", "#DC2626"])
axes[0].set_title("Resolution Status")
axes[0].set_xlabel("resolution_bool")
axes[0].set_ylabel("events")

pd.to_numeric(closed["resolution"], errors="coerce").plot.hist(ax=axes[1], bins=40, color="#059669", alpha=0.85)
axes[1].set_title("Resolution Time for Closed Events")
axes[1].set_xlabel("hours")
plt.show()

In [ ]:
results, predictions, feature_importance = run_resolution_time_ml(observed)

display(compact_modeling_summary(results, ["mae", "rmse", "r2"]).head(30))
display(results.sort_values(["mae", "rmse"], ascending=[True, True], kind="stable").head(30))
display(feature_importance.head(30))

print(f"saved: {RESOLUTION_TIME_RESULTS_PATH}")
print(f"saved: {RESOLUTION_PREDICTIONS_PATH}")
print(f"saved: {RESOLUTION_FEATURE_IMPORTANCE_PATH}")
print(f"saved: {RESOLUTION_BIAS_DIAGNOSTICS_PATH}")
print("incremental checkpoints: data/processed/modeling/diagnostics/modeling_stage/checkpoints/resolution_time")

In [ ]:
results = pd.read_csv(RESOLUTION_TIME_RESULTS_PATH)
predictions = pd.read_parquet(RESOLUTION_PREDICTIONS_PATH)
feature_importance = pd.read_csv(RESOLUTION_FEATURE_IMPORTANCE_PATH)
bias = pd.read_csv(RESOLUTION_BIAS_DIAGNOSTICS_PATH)
leakage_audit = pd.read_csv(RESOLUTION_LEAKAGE_AUDIT_PATH)

metric_cols = [
    "rmse",
    "mae",
    "r2",
    "spearman_correlation",
    "pearson_correlation",
    "median_absolute_error",
    "train_mae",
    "validation_mae",
    "test_mae",
    "cv_validation_primary_score",
    "train_vs_validation_gap",
    "validation_vs_test_gap",
    "possible_overfitting",
    "possible_test_degradation",
]
display(results[["model_name", "split_strategy", "status"] + [c for c in metric_cols if c in results.columns]].query("status == 'ok'"))
display(bias.head(50))
display(leakage_audit[leakage_audit["excluded_from_predictors"]].head(30))

## Archetype and Anomaly Error Diagnostics

The cluster output is especially useful for resolution-time inference because some event regimes may create larger service delays.
We use clusters and anomaly scores for diagnostics after prediction, not as the default regression target construction.

In [ ]:
CLUSTER_PATH = ROOT / "data" / "processed" / "modeling" / "clustering_event_archetypes.parquet"
ANOMALY_PATH = ROOT / "data" / "processed" / "modeling" / "anomaly_event_scores.parquet"

if CLUSTER_PATH.exists() and ANOMALY_PATH.exists():
    clusters = pd.read_parquet(CLUSTER_PATH)[
        ["event_id", "combined_cluster_id", "cluster_label"]
    ].copy()
    anomalies = pd.read_parquet(ANOMALY_PATH)[
        ["event_id", "anomaly_score", "anomaly_flag", "anomaly_method_agreement"]
    ].copy()
    resolution_cluster_diag = (
        predictions.merge(clusters, on="event_id", how="left")
        .merge(anomalies, on="event_id", how="left")
    )
    resolution_cluster_diag = resolution_cluster_diag[resolution_cluster_diag["set_name"].eq("test")].copy()
    resolution_cluster_diag["residual"] = resolution_cluster_diag["y_true"] - resolution_cluster_diag["y_pred"]
    resolution_cluster_diag["absolute_error"] = resolution_cluster_diag["residual"].abs()
    resolution_by_cluster = (
        resolution_cluster_diag.groupby(["combined_cluster_id", "cluster_label"], dropna=False)
        .agg(
            n_events=("event_id", "size"),
            mae_hours=("absolute_error", "mean"),
            mean_residual_hours=("residual", "mean"),
            underprediction_rate=("residual", lambda s: pd.Series(s).gt(0).mean()),
            mean_anomaly_score=("anomaly_score", "mean"),
            anomaly_share=("anomaly_flag", lambda s: pd.Series(s).fillna(False).astype(bool).mean()),
        )
        .reset_index()
        .sort_values("mae_hours", ascending=False, kind="stable")
    )
    resolution_by_cluster.to_csv(
        ROOT / "data" / "processed" / "modeling" / "diagnostics_resolution_by_cluster.csv",
        index=False,
    )
    display(resolution_by_cluster)
else:
    print("Cluster/anomaly outputs are not available yet. Run 13_clustering-anomaly first.")

In [ ]:
best_row = (
    results.query("status == 'ok'")
    .sort_values(["mae", "rmse"], ascending=[True, True], kind="stable")
    .iloc[0]
)
best_predictions = predictions[
    (predictions["model_name"] == best_row["model_name"])
    & (predictions["split_strategy"] == best_row["split_strategy"])
    & (predictions["set_name"] == "test")
].copy()
best_predictions["residual"] = best_predictions["y_true"] - best_predictions["y_pred"]

fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
axes[0, 0].scatter(best_predictions["y_true"], best_predictions["y_pred"], alpha=0.5, color="#059669")
axes[0, 0].plot(
    [best_predictions["y_true"].min(), best_predictions["y_true"].max()],
    [best_predictions["y_true"].min(), best_predictions["y_true"].max()],
    linestyle="--",
    color="#64748B",
)
axes[0, 0].set_title("Observed vs Predicted Resolution Time")
axes[0, 0].set_xlabel("observed hours")
axes[0, 0].set_ylabel("predicted hours")

axes[0, 1].hist(pd.to_numeric(best_predictions["residual"], errors="coerce").dropna(), bins=40, color="#DC2626", alpha=0.85)
axes[0, 1].set_title("Residual Distribution")
axes[0, 1].set_xlabel("observed - predicted hours")

top_features = feature_importance.head(20).sort_values("importance", kind="stable")
axes[1, 0].barh(top_features["feature"], top_features["importance"], color="#2563EB")
axes[1, 0].set_title("Top Feature Importances")
axes[1, 0].set_xlabel("importance")

bias.query("bias_dimension == 'borough'").plot.barh(x="group_value", y="mae", ax=axes[1, 1], color="#7C3AED", legend=False)
axes[1, 1].set_title("Resolution MAE by Borough")
axes[1, 1].set_xlabel("MAE hours")
plt.show()